# Basis recombination

This section describes the basis recombination matrices for Dirichlet and Neumann boundary conditions.

## Theory

When Chebyshev series are used to solve boundary-value problems, it is often convenient to work with a recombined basis $\{\phi_n\}_{n=0}^{N}$, built as linear combinations of the Chebyshev polynomials $\{T_n\}_{n=0}^{N}$, chosen so that most recombined functions automatically satisfy the boundary condition. A function expanded on the recombined basis,
```{math}
f(x) = \sum_{n=0}^{N} \tilde{f}_n \, \phi_n(x) \;,
```
then only needs its first one or two coefficients adjusted to impose the boundary values, while the rest of the expansion is free to satisfy the equation in the interior. The recombination is expressed as a matrix $\mat{R}$ so that $\phi_n(x) = \sum_m R_{mn} T_m(x)$, i.e. column $n$ of $\mat{R}$ holds the coefficients of $\phi_n$ on the original Chebyshev basis.

### Dirichlet recombination

The `dirichlet` matrix builds a basis that is zero at both ends of the interval, except for the first two functions:
```{math}
:label: dirichlet_basis
\phi_0(x) = \tfrac{1}{2}\bigl[T_0(x) - T_1(x)\bigr] \;, \quad
\phi_1(x) = \tfrac{1}{2}\bigl[T_0(x) + T_1(x)\bigr] \;, \quad
\phi_n(x) = \tfrac{1}{2}\bigl[T_n(x) - T_{n-2}(x)\bigr] \; (n \geq 2) \;.
```
Using $T_n(a)=(-1)^n$ and $T_n(b)=1$ (through the mapping {eq}`xi_map`, where $\xi=-1$ corresponds to $x=a$ and $\xi=+1$ to $x=b$), one finds
```{math}
\phi_0(a)=1 \;,\quad
\phi_0(b)=0 \;,\quad
\phi_1(a)=0 \;,\quad
\phi_1(b)=1 \;,\quad\text{and }
\phi_n(a)=\phi_n(b)=0\text{ for }n \geq 2
\;.
```
A function $f$ therefore takes the prescribed boundary values $f(a)=\tilde{f}_0$, $f(b)=\tilde{f}_1$ regardless of the remaining coefficients, which makes it straightforward to enforce Dirichlet boundary conditions in a spectral discretisation.

### Neumann recombination

The `neumann` matrix instead builds a basis with zero *derivative* at both ends, except for the first and second functions, which carry the boundary derivatives:
```{math}
\phi_0'(a)=1 \;,\quad
\phi_0'(b)=0 \;,\quad
\phi_1'(a)=0 \;,\quad
\phi_1'(b)=1 \;,\quad
\phi_n'(a)=\phi_n'(b)=0\text{ for }n \geq 2.
```
The first three functions in this basis are defined as:
```{math}
:label: neumann_basis
\phi_0(x) = \tfrac{1}{2}T_1(x) - \tfrac{1}{8} T_2(x) \;, \quad
\phi_1(x) = \tfrac{1}{2}T_1(x) + \tfrac{1}{8} T_2(x) \;, \quad
\phi_2(x) = T_0(x) \;.
```
The other functions are defined as 
```{math}
\phi_n(x) = 
\begin{cases}
\tfrac{1}{n^2}T_n(x) - T_1(x) & n \text{ odd} \;, \\
\tfrac{1}{n^2}T_n(x) - \tfrac{1}{4} T_2(x) & n \text{ even} \;.
\end{cases}
```

## Python API

The `Basis1D` class provides the methods `dirichlet` and `neumann`, which return the square recombination matrix $\mat{R}$ described above. Given the coefficients $\tilde{\vec{c}}$ of a recombined function, the coefficients $\vec{c}$ on the ordinary Chebyshev basis are recovered simply as $\vec{c} = \mat{R}\vec{\tilde{c}}$, which can then be evaluated with `Basis1D.eval` or wrapped in a `RealFunction`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%config InlineBackend.figure_formats = ["svg", "pdf"]
from cheby import Basis1D

basis = Basis1D(6, -1, 1)
R_dir = basis.dirichlet()
R_neu = basis.neumann()

x = np.linspace(-1, 1, 300)
Tn = basis.eval(x)

phi_dir = Tn @ R_dir
phi_neu = Tn @ R_neu

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(x, phi_dir)
axes[0].set_title('Dirichlet recombination')
axes[0].set_xlabel(r'$x$')
axes[1].plot(x, phi_neu)
axes[1].set_title('Neumann recombination')
axes[1].set_xlabel(r'$x$')
plt.tight_layout()
plt.show()

All the Dirichlet-recombined functions vanish at $x=\pm1$ except $\phi_0$ and $\phi_1$, which take the values $1$ and $0$ (or $0$ and $1$) at the two ends. Similarly, all Neumann-recombined functions have zero slope at $x=\pm1$ except $\phi_0$ and $\phi_1$. This can be checked directly from the derivative matrix of {doc}`derivative`:

In [ ]:
D = basis.diff_matrix()
dphi_neu_ends = (D @ R_neu)[[0, -1], :]
T_ends = basis.eval(np.array([-1.0, 1.0]))
print(np.round(T_ends @ (D @ R_neu), 6))

Each column above gives the derivative of the corresponding recombined function at $x=-1$ (first row) and $x=1$ (second row): only columns 1 and 2 (the second and third basis functions) are non-zero, as expected.

## References

```{bibliography}
:filter: docname in docnames
```